# Tools, Skills, and the AI as Conductor
## ME 493B — AI in Product Development | Mini-Project 3, Part A

**Instructor:** Scott Thielman, PhD — University of Washington Bothell
**Due:** Monday, May 11, 2026 at 11:59 PM
**Time estimate:** 90–120 minutes
**Points:** 50 (Part A). Part B is worth 50 points separately.

---

### What changed

In MP1 you saw what the AI knows. In MP2 you fed it what *you* know. In MP3
you give it the ability to **do things** — call your code, follow your
workflows, look at your designs. The AI stops being a chatbot and becomes
an orchestrator that uses tools you build to accomplish goals you set.

This notebook teaches you to expose capability to the AI:

1. **Function calling (tool use)** — the AI decides when to invoke your code
2. **Skill authoring** — codifying engineering judgment as a reusable artifact
3. **RAG-as-a-tool** — productizing your MP2 retrieval so the AI can call it
4. **Vision in the loop** — host-based design review (no API plumbing)
5. **Composition** — a skill that orchestrates multiple tools at once

In Part B, you'll assemble these into a workspace for the MiniClaw project.

### How visibility works in this notebook

Function calling is invisible by default — the AI calls a tool, you only see
the final text. That makes it impossible to grade and impossible to debug.

The Setup cell defines a helper, `run_with_logging()`, that wraps the API
call and prints a structured transcript every time you run a query:

```
[USER] ...
[ITERATION 1]
[TOOL CALL] convert_units({...})
[TOOL RESULT] ...
[ITERATION 2]
[AI FINAL RESPONSE] ...
```

**Sections 1, 2, 3, and 5 are graded from these transcripts.** If your
transcript shows `[TOOL CALL]` lines with the right tools and arguments,
you've done the work. **Section 4 is graded host-side** — markdown +
screenshot + response text, the same evidence pattern Part B uses for the
CAD interaction.

### Grading summary (50 pts)

| Section | Points | What the grader checks |
|---------|--------|------------------------|
| 1. Function Calling | 8 | Three transcripts show AI deciding correctly when to call the tool |
| 2. Anatomy of a Skill | 8 | Student-authored skill exists, transcript shows AI catching unit error |
| 3. Wrap Your RAG as a Tool | 10 | Tool schema printed, three transcripts, vague-vs-good comparison |
| 4. Vision in the Loop | 8 | Three documented exchanges (markdown + screenshot + response) |
| 5. Composition Capstone | 8 | Composite skill exists, with/without skill comparison |
| 6. Reflections | 8 | Three thoughtful reflections (3–4 sentences each) |
| **Total** | **50** | |

> **Reminder:** This notebook builds on MP2's GitHub Models setup. You
> already have a Personal Access Token with `models:read` scope in your
> `.env` file. If `setup` complains about credentials, refer back to your
> MP2 PAT setup notes.

---
## Section 0: Setup

Run the cells in this section **before anything else**. They install
dependencies, define `run_with_logging()`, and load the starter RAG. These
are the only pre-written code cells in the notebook — everything in
Sections 1–5 is yours to write.

In [1]:
# ============================================================
# SETUP — Run this cell first (pre-written, do not modify)
# ============================================================
import subprocess, sys

# All MP3 Part A dependencies should already be in requirements.txt from
# MP2 — but we double-check the few we depend on here.
for pkg in ["openai", "chromadb", "sentence-transformers", "python-dotenv"]:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        stdout=subprocess.DEVNULL,
    )

import json
import os
import textwrap
import time
from datetime import datetime
from pathlib import Path

from openai import OpenAI
from dotenv import load_dotenv

# Load .env (your GitHub PAT lives here — same as MP2)
load_dotenv()

# GitHub Models endpoint — OpenAI-compatible, free tier
GH_TOKEN = os.environ.get("GITHUB_TOKEN") or os.environ.get("GITHUB_PAT")
if not GH_TOKEN:
    raise RuntimeError(
        "Could not find GITHUB_TOKEN or GITHUB_PAT in environment. "
        "Make sure you have a .env file with GITHUB_TOKEN=ghp_... at "
        "the repo root. See your MP2 PAT setup notes."
    )

client = OpenAI(
    base_url="https://models.github.ai/inference",
    api_key=GH_TOKEN,
)

# Default model. We pin a single tool-calling SKU here. If GitHub Models
# rotates the catalog (it does, every few months), swap this for the
# current verified-stable tool-calling model — the lesson is the pattern.
DEFAULT_MODEL = "openai/gpt-4.1"

# Where transcripts get written by run_with_logging() (Sections 1, 2, 3, 5)
TRANSCRIPTS_DIR = Path("transcripts")
if not TRANSCRIPTS_DIR.exists():
    # Handle running from repo root vs. from notebook directory
    TRANSCRIPTS_DIR = Path("MP3/Part A/transcripts")
TRANSCRIPTS_DIR.mkdir(exist_ok=True)

print("Imports loaded.")
print(f"  Default model: {DEFAULT_MODEL}")
print(f"  Transcripts will be written to: {TRANSCRIPTS_DIR.resolve()}")


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


Imports loaded.
  Default model: openai/gpt-4.1
  Transcripts will be written to: /workspaces/ai-in-pd-spring2026-2/MP3/Part A/transcripts


In [2]:
# ============================================================
# run_with_logging() — the visibility mechanism for this notebook
#
# Wraps a multi-turn GitHub Models call. Executes any tool calls the AI
# makes, prints a structured transcript, and returns a dict you can
# inspect programmatically.
#
# READ THIS CAREFULLY — you'll use it in every section. Understanding
# the agent loop is part of the lesson.
# ============================================================

def run_with_logging(messages, *, tools=None, tool_handlers=None,
                     model=None, system=None, max_iterations=5,
                     verbose=True, label=None):
    """Run a multi-turn tool-using conversation, printing every step.

    Args:
        messages: list of {"role": "user"/"assistant", "content": str}
        tools: list of OpenAI-style tool schema dicts, or None
        tool_handlers: dict mapping tool name → Python callable
        model: model identifier (defaults to DEFAULT_MODEL above)
        system: optional system prompt prepended as a "system" message
        max_iterations: safety cap on agent-loop iterations
        verbose: if True, print structured transcript to cell output
        label: optional short label used in the saved JSON filename

    Returns:
        dict with keys:
            "final_message":   the AI's final text response
            "tool_calls_made": list of {"name", "args", "result"} dicts
            "iterations":      number of loop iterations used
            "transcript":      the full printed transcript as a string
    """
    model = model or DEFAULT_MODEL
    tool_handlers = tool_handlers or {}

    # Build the message list (mutates as the loop runs)
    full_messages = []
    if system:
        full_messages.append({"role": "system", "content": system})
    full_messages.extend(messages)

    # We collect printed lines into a buffer so we can save the transcript
    lines = []
    def out(s=""):
        lines.append(s)
        if verbose:
            print(s)

    bar = "═" * 62
    out(bar)
    # Print the user's question (the LAST user message — concise)
    last_user = next((m["content"] for m in reversed(messages) if m["role"] == "user"), "")
    out(f"[USER]\n{last_user}")
    out(bar)

    tool_calls_made = []
    final_text = None

    for iteration in range(1, max_iterations + 1):
        out(f"[ITERATION {iteration}]")

        kwargs = {"model": model, "messages": full_messages}
        if tools:
            kwargs["tools"] = tools
            kwargs["tool_choice"] = "auto"

        response = client.chat.completions.create(**kwargs)
        msg = response.choices[0].message

        # Did the AI ask to call any tools?
        tcs = getattr(msg, "tool_calls", None) or []

        if tcs:
            # Append the assistant message that contained the tool calls
            full_messages.append({
                "role": "assistant",
                "content": msg.content or "",
                "tool_calls": [
                    {
                        "id": tc.id,
                        "type": "function",
                        "function": {
                            "name": tc.function.name,
                            "arguments": tc.function.arguments,
                        },
                    }
                    for tc in tcs
                ],
            })

            for tc in tcs:
                name = tc.function.name
                try:
                    args = json.loads(tc.function.arguments or "{}")
                except json.JSONDecodeError:
                    args = {"_raw_arguments": tc.function.arguments}

                out(f"[TOOL CALL] {name}")
                out(f"  Args: {json.dumps(args, ensure_ascii=False)}")

                handler = tool_handlers.get(name)
                if handler is None:
                    result = {"error": f"No handler registered for tool '{name}'"}
                else:
                    try:
                        result = handler(**args)
                    except Exception as e:  # noqa: BLE001
                        result = {"error": f"{type(e).__name__}: {e}"}

                out(f"[TOOL RESULT]")
                pretty = json.dumps(result, ensure_ascii=False, indent=2)
                # Truncate long results in the printed view (full result is
                # still in the saved JSON transcript)
                if len(pretty) > 600:
                    out(textwrap.indent(pretty[:600] + " ...", "  "))
                else:
                    out(textwrap.indent(pretty, "  "))

                tool_calls_made.append({"name": name, "args": args, "result": result})

                full_messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": json.dumps(result, ensure_ascii=False),
                })

            out(bar)
            continue  # back to the model with the tool results in context

        # No tool calls → this is the final answer
        final_text = msg.content or ""
        out("[AI FINAL RESPONSE]")
        out(final_text)
        out(bar)
        break
    else:
        out(f"[MAX ITERATIONS HIT after {max_iterations} loops — stopping]")
        out(bar)

    out("[SUMMARY]")
    out(f"  Iterations: {iteration}")
    names = ", ".join(t["name"] for t in tool_calls_made) or "(none)"
    out(f"  Tool calls made: {len(tool_calls_made)} ({names})")
    out(f"  Final answer received: {'yes' if final_text else 'no'}")

    transcript = "\n".join(lines)

    # Save a machine-readable copy of the run
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    slug = (label or "run").lower().replace(" ", "_").replace("/", "_")
    out_path = TRANSCRIPTS_DIR / f"{slug}_{ts}.json"
    out_path.write_text(json.dumps({
        "label": label,
        "model": model,
        "messages_in": messages,
        "system": system,
        "tools_offered": [t.get("function", {}).get("name") for t in (tools or [])],
        "tool_calls_made": tool_calls_made,
        "iterations": iteration,
        "final_message": final_text,
        "transcript": transcript,
    }, ensure_ascii=False, indent=2), encoding="utf-8")

    return {
        "final_message": final_text,
        "tool_calls_made": tool_calls_made,
        "iterations": iteration,
        "transcript": transcript,
        "transcript_path": str(out_path),
    }

# Quick helper for visualizing the JSON tool schemas you'll write
def pretty_print_tool_schema(schema):
    print(json.dumps(schema, indent=2, ensure_ascii=False))

print("run_with_logging() and pretty_print_tool_schema() are defined.")

run_with_logging() and pretty_print_tool_schema() are defined.


In [3]:
# ============================================================
# Starter RAG — Ridgeline Engineering Partners corpus from MP2
# ============================================================
# The starter_rag/ folder ships with a pre-built ChromaDB collection
# (109 chunks across 20 Ridgeline documents). The query helper below is
# what you'll wrap as a tool in Section 3.
import sys
starter_dir = Path("starter_rag")
if not starter_dir.exists():
    starter_dir = Path("MP3/Part A/starter_rag")
sys.path.insert(0, str(starter_dir.resolve()))

# If the ChromaDB hasn't been built yet on this machine, build it now.
if not (starter_dir / "chroma_db").exists():
    print("Starter RAG ChromaDB not found — building it now (one-time, ~30s) ...")
    import importlib
    builder = importlib.import_module("build_starter_rag")
    builder.build()

from query_ridgeline_rag import query_ridgeline_rag

# Smoke test
smoke = query_ridgeline_rag("What is the standard billing rate for senior engineers?", n_results=2)
print("Starter RAG ready.")
print(f"  {smoke['summary']}")
print(f"  Top chunk preview: {smoke['chunks'][0]['text'][:140]!r} ...")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starter RAG ready.
  2 chunks retrieved. Top match: DOC-014 — Standard Billing Rate Schedule — FY 2025 (score 0.6404).
  Top chunk preview: 'RIDGELINE ENGINEERING PARTNERS\nSTANDARD BILLING RATE SCHEDULE\nEffective: January 1, 2025\nApproved by: Dr. Catherine Marsh, Managing Partner\n' ...


---
## Section 1: Function Calling — How the AI Calls Your Code [8 pts]

**Concept.** Function calling is the mechanism by which the AI decides to
invoke external code. You define a tool with a JSON schema, send it
alongside the user's message, and the AI either responds normally OR
returns a structured request to call the tool. Your code executes,
returns the result, and the AI continues with the result in context.
**The AI is a decider; the tool is just Python.**

The agent loop:

```
user → AI → (optional tool call) → tool result → AI → final answer
```

Multi-turn loops are normal. The AI may call a tool, look at the result,
and decide to call another (or the same) tool again before giving you a
final answer.

### What to build

1. Wrap the provided unit-conversion math in a Python function
   `convert_units(value, from_unit, to_unit)` that returns a dict.
2. Write the JSON schema for the tool — `name`, `description`, and
   `parameters` with `type`, `properties`, `required`.
3. Run three queries through `run_with_logging()`:
   - **Query A** (should call tool): *"What's 25.4 mm in inches?"*
   - **Query B** (should NOT call tool): *"What's the typical Young's modulus of mild steel?"*
   - **Query C** (judgment call): *"I'm machining a part with a 0.001 inch
     tolerance — what's that in millimeters, and is that achievable on a
     hobby mill?"*  (needs the tool *and* domain knowledge)
4. Print a summary table comparing `tool_calls_made` vs. expectation.

### Key concepts

- The JSON schema is a **contract**. The AI reads it to decide if your
  tool is relevant.
- A good `description` is the difference between the AI calling your
  tool and ignoring it. Vague descriptions get ignored; specific ones
  get used.
- Tool choice is the AI's responsibility. You expose capability; the AI
  decides whether to use it.

### Reference links

- [OpenAI function-calling guide](https://platform.openai.com/docs/guides/function-calling)
- [GitHub Models tool use](https://docs.github.com/en/github-models)

### Proof of work [8 pts]

Three labeled `run_with_logging()` transcripts (A, B, C) printed in cell
output, a summary table that compares `tool_calls_made` to the expected
behavior, and a 2-sentence student observation: *"What changed about
the AI's behavior when you gave it a tool?"*

In [4]:
# Step 1 — Wrap the conversion math as a Python function.
#
# Use the table below. Each pair is bidirectional (mm↔inch, MPa↔psi, etc.).
# Your function should return a dict like:
#     {"value_in": <input>, "from_unit": ..., "to_unit": ...,
#      "value_out": <converted>, "factor": <multiplier>}
#
# Conversion factors (forward direction; invert for the reverse):
#     1 inch    = 25.4 mm
#     1 psi     = 0.00689476 MPa
#     1 in·lb   = 0.112985 N·m
#     1 lbf     = 4.44822 N
#     °F        = °C * 9/5 + 32
#
# Raise ValueError for unit pairs that aren't supported.
#
# def convert_units(value: float, from_unit: str, to_unit: str) -> dict:
#     ...
#
# Test it on a couple of known cases (e.g., 25.4 mm → inches) before moving on.
with open("unit_conversion_tool.py", "w", encoding="utf-8") as f:
    f.write(textwrap.dedent("""
    def convert_units(value: float, from_unit: str, to_unit: str) -> dict:
        if from_unit == "mm" and to_unit == "inch":
            factor = 1 / 25.4
            value_out = value * factor
        elif from_unit == "inch" and to_unit == "mm":
            factor = 25.4
            value_out = value * factor
        elif from_unit == "psi" and to_unit == "MPa":
            factor = 0.00689476
            value_out = value * factor
        elif from_unit == "MPa" and to_unit == "psi":
            factor = 1 / 0.00689476
            value_out = value * factor
        elif from_unit == "in·lb" and to_unit == "N·m":
            factor = 0.112985
            value_out = value * factor
        elif from_unit == "N·m" and to_unit == "in·lb":
            factor = 1 / 0.112985
            value_out = value * factor
        elif from_unit == "lbf" and to_unit == "N":
            factor = 4.44822
            value_out = value * factor
        elif from_unit == "N" and to_unit == "lbf":
            factor = 1 / 4.44822
            value_out = value * factor
        elif from_unit == "°C" and to_unit == "°F":
            factor = None  # Not a simple multiplier
            value_out = value * 9/5 + 32
        elif from_unit == "°F" and to_unit == "°C":
            factor = None  # Not a simple multiplier
            value_out = (value - 32) * 5/9
        else:
            raise ValueError(f"Unsupported unit conversion: {from_unit} to {to_unit}")

        return {
            "value_in": value,
            "from_unit": from_unit,
            "to_unit": to_unit,
            "value_out": value_out,
            "factor": factor,
        }
    """).strip())
print("Unit conversion function defined in unit_conversion_tool.py.")

#testing
from unit_conversion_tool import convert_units
test_cases = [
    (25.4, "mm", "inch"),
    (1, "inch", "mm"),
    (100, "psi", "MPa"),
    (0.00689476, "MPa", "psi"),
    (10, "in·lb", "N·m"),
    (1.12985, "N·m", "in·lb"),
    (5, "lbf", "N"),
    (22.2411, "N", "lbf"),
    (100, "°C", "°F"),
    (212, "°F", "°C"),
]
for value, from_u, to_u in test_cases:
    result = convert_units(value, from_u, to_u)
    print(f"{value} {from_u} → {result['value_out']} {to_u} (factor: {result['factor']})")

Unit conversion function defined in unit_conversion_tool.py.
25.4 mm → 0.9999999999999999 inch (factor: 0.03937007874015748)
1 inch → 25.4 mm (factor: 25.4)
100 psi → 0.689476 MPa (factor: 0.00689476)
0.00689476 MPa → 1.0 psi (factor: 145.0376807894691)
10 in·lb → 1.12985 N·m (factor: 0.112985)
1.12985 N·m → 10.0 in·lb (factor: 8.850732398105944)
5 lbf → 22.2411 N (factor: 4.44822)
22.2411 N → 5.0 lbf (factor: 0.2248090247334889)
100 °C → 212.0 °F (factor: None)
212 °F → 100.0 °C (factor: None)


In [5]:
# Step 2 — Write the JSON schema for convert_units.
#
# The schema is a Python dict that follows the OpenAI tool format:
#     {
#       "type": "function",
#       "function": {
#         "name": ...,
#         "description": ...,   # be specific — this is what the AI reads
#         "parameters": {
#           "type": "object",
#           "properties": {
#             "value": {"type": "number", "description": "..."},
#             "from_unit": {"type": "string", "description": "..."},
#             "to_unit":   {"type": "string", "description": "..."},
#           },
#           "required": ["value", "from_unit", "to_unit"]
#         }
#       }
#     }
#
# Save it as `convert_units_tool` and print it with pretty_print_tool_schema().
#
# convert_units_tool = { ... }
# pretty_print_tool_schema(convert_units_tool)
convert_units_tool = {
    "type": "function",
    "function": {
        "name": "convert_units",
        "description": (
            "Convert a numerical value from one unit to another. Supported "
            "units are mm, inch, psi, MPa, in·lb, N·m, lbf, N, °C, and °F. "
            "The function returns the converted value along with the input "
            "value and units."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "value": {
                    "type": "number",
                    "description": "The numerical value to convert."
                },
                "from_unit": {
                    "type": "string",
                    "description": (
                        "The unit of the input value. Supported units: mm, inch, "
                        "psi, MPa, in·lb, N·m, lbf, N, °C, °F."
                    )
                },
                "to_unit": {
                    "type": "string",
                    "description": (
                        "The unit to convert to. Supported units: mm, inch, "
                        "psi, MPa, in·lb, N·m, lbf, N, °C, °F."
                    )
                },
            },
            "required": ["value", "from_unit", "to_unit"]
        }
    }
}
print("Unit conversion tool schema defined:")
pretty_print_tool_schema(convert_units_tool)

Unit conversion tool schema defined:
{
  "type": "function",
  "function": {
    "name": "convert_units",
    "description": "Convert a numerical value from one unit to another. Supported units are mm, inch, psi, MPa, in·lb, N·m, lbf, N, °C, and °F. The function returns the converted value along with the input value and units.",
    "parameters": {
      "type": "object",
      "properties": {
        "value": {
          "type": "number",
          "description": "The numerical value to convert."
        },
        "from_unit": {
          "type": "string",
          "description": "The unit of the input value. Supported units: mm, inch, psi, MPa, in·lb, N·m, lbf, N, °C, °F."
        },
        "to_unit": {
          "type": "string",
          "description": "The unit to convert to. Supported units: mm, inch, psi, MPa, in·lb, N·m, lbf, N, °C, °F."
        }
      },
      "required": [
        "value",
        "from_unit",
        "to_unit"
      ]
    }
  }
}


In [6]:
# Step 3 — Query A: should call the tool
#
# Use run_with_logging() with:
#   messages       = [{"role": "user", "content": "What's 25.4 mm in inches?"}]
#   tools          = [convert_units_tool]
#   tool_handlers  = {"convert_units": convert_units}
#   label          = "section1_queryA"
#
# Inspect the transcript: did the AI call convert_units?
# result_A = run_with_logging( ... )
result_A = run_with_logging(
    messages=[{"role": "user", "content": "What's 25.4 mm in inches?"}],
    tools=[convert_units_tool],
    tool_handlers={"convert_units": convert_units},
    label="section1_queryA"
)


══════════════════════════════════════════════════════════════
[USER]
What's 25.4 mm in inches?
══════════════════════════════════════════════════════════════
[ITERATION 1]
[TOOL CALL] convert_units
  Args: {"from_unit": "mm", "to_unit": "inch", "value": 25.4}
[TOOL RESULT]
  {
    "value_in": 25.4,
    "from_unit": "mm",
    "to_unit": "inch",
    "value_out": 0.9999999999999999,
    "factor": 0.03937007874015748
  }
══════════════════════════════════════════════════════════════
[ITERATION 2]
[AI FINAL RESPONSE]
25.4 mm is exactly 1 inch.
══════════════════════════════════════════════════════════════
[SUMMARY]
  Iterations: 2
  Tool calls made: 1 (convert_units)
  Final answer received: yes


In [7]:
# Step 3 — Query B: should NOT call the tool
# Same setup, different question:
#   "What's the typical Young's modulus of mild steel?"
# label = "section1_queryB"
result_B = run_with_logging(
    messages=[{"role": "user", "content": "What's the typical Young's modulus of mild steel?"}],
    tools=[convert_units_tool],
    tool_handlers={"convert_units": convert_units},
    label="section1_queryB"
)


══════════════════════════════════════════════════════════════
[USER]
What's the typical Young's modulus of mild steel?
══════════════════════════════════════════════════════════════
[ITERATION 1]
[AI FINAL RESPONSE]
The typical Young's modulus of mild steel is around **200 GPa** (gigapascals), which is equivalent to **200,000 MPa** or **29 million psi**. This value may vary slightly depending on the specific composition and processing, but 200 GPa is generally accepted for engineering calculations.
══════════════════════════════════════════════════════════════
[SUMMARY]
  Iterations: 1
  Tool calls made: 0 ((none))
  Final answer received: yes


In [8]:
# Step 3 — Query C: judgment call (tool + domain knowledge)
# Question:
#   "I'm machining a part with a 0.001 inch tolerance — what's that in
#    millimeters, and is that achievable on a hobby mill?"
# label = "section1_queryC"
result_C = run_with_logging(
    messages=[{"role": "user", "content": (
        "I'm machining a part with a 0.001 inch tolerance — what's that in "
        "millimeters, and is that achievable on a hobby mill?"
    )}],
    tools=[convert_units_tool],
    tool_handlers={"convert_units": convert_units},
    label="section1_queryC"
)

══════════════════════════════════════════════════════════════
[USER]
I'm machining a part with a 0.001 inch tolerance — what's that in millimeters, and is that achievable on a hobby mill?
══════════════════════════════════════════════════════════════
[ITERATION 1]
[TOOL CALL] convert_units
  Args: {"from_unit": "inch", "to_unit": "mm", "value": 0.001}
[TOOL RESULT]
  {
    "value_in": 0.001,
    "from_unit": "inch",
    "to_unit": "mm",
    "value_out": 0.0254,
    "factor": 25.4
  }
══════════════════════════════════════════════════════════════
[ITERATION 2]
[AI FINAL RESPONSE]
A 0.001 inch tolerance is equivalent to 0.0254 millimeters (25.4 microns).

As for achievability on a hobby mill:
- 0.001" (0.0254 mm) is considered a very tight tolerance, often reserved for precision machining (tool rooms, CNCs, surface grinding).
- Most hobby mills (manual or entry-level CNCs) can reliably hold tolerances around 0.002–0.005" (0.05–0.13 mm); achieving 0.001" consistently may be challenging w

In [9]:
# Step 4 — Summary table.
#
# Print something like:
#   Query A (mm to inches):           tool_calls_made = 1   [expected: 1]
#   Query B (Young's modulus):        tool_calls_made = 0   [expected: 0]
#   Query C (tolerance + judgment):   tool_calls_made = ?   [expected: 1]
#
# Use the result["tool_calls_made"] lists from the runs above.
print("Summary of tool calls made:")
print(f"  Query A (mm to inches):           tool_calls_made = {len(result_A['tool_calls_made'])}   [expected: 1]")
print(f"  Query B (Young's modulus):        tool_calls_made = {len(result_B['tool_calls_made'])}   [expected: 0]")
print(f"  Query C (tolerance + judgment):   tool_calls_made = {len(result_C['tool_calls_made'])}   [expected: 1]")

Summary of tool calls made:
  Query A (mm to inches):           tool_calls_made = 1   [expected: 1]
  Query B (Young's modulus):        tool_calls_made = 0   [expected: 0]
  Query C (tolerance + judgment):   tool_calls_made = 1   [expected: 1]


### ✏️ YOUR OBSERVATION

*In 2–3 sentences, answer:* what changed about the AI's behavior when
you gave it a tool? Did anything surprise you about Query C?

> It seems to become more consistent, it strictly follows the rules of the tool and uses it to a higher degree of precision that it could reach simply attempting to find answers on its own. Query C suprised me because the AI automatically detected it should break down the problem into 2 steps, one for each question, first converting units then using its general knowledge to give feedback on the feasability of that tolerance.

> **Forward connection.** The same mechanism scales to any Python
> function — calculation, file I/O, web requests, database queries. In
> Section 3 you'll wrap your RAG retrieval as a tool and watch the AI
> choose to query your engineering knowledge base. In Section 5 you'll
> compose this unit converter with the RAG tool to handle questions
> that need both.

---
## Section 2: Anatomy of a Skill [8 pts]

**Concept.** A *skill* is a reusable orchestration artifact — a markdown
file that tells an AI how to handle a class of tasks. Skills capture
the "how to do this kind of work" knowledge that an experienced
engineer has but doesn't write down. By codifying it, you make that
judgment available to **any AI session, by anyone**, without you being
there.

Skills sit *between* system prompts and tools:

- **Above tools** — they orchestrate which tools to call when
- **Below system prompts** — they're reusable across sessions; system
  prompts vary per session

The format isn't standardized across the industry yet. Anthropic uses
`SKILL.md`, OpenAI uses Custom GPT instructions, Cursor uses
`.cursorrules`, GitHub Copilot uses `.github/copilot-instructions.md`.
The *idea* is the same — codified, reusable orchestration.

### What to build

1. **Read** `SKILL_me493b_course_structure.md` at the repo root. Identify
   its anatomy (front matter, "when to use," structured guidance,
   instructions, anti-patterns).
2. **Read** the provided specimen `skills/example_design_review.md`.
   Compare its structure to the course skill.
3. **Write your own** small skill: `skills/unit_check_skill.md`. The
   skill should help an AI catch unit-conversion errors in engineering
   text. Required elements:
   - YAML front matter (`name`, `description`)
   - "When to use this skill" — the trigger conditions
   - "Steps" — the workflow (3–5 steps)
   - "What to flag" — specific patterns to watch for
   - "What NOT to do" — at least one anti-pattern
4. **Test it.** Load the skill text as a system prompt and send a test
   message containing a unit error:
   > *"The shaft transmits 50 N·m of torque with a safety factor of
   > 2.5, requiring a diameter of 12 inches per Lewis bending equation."*
   Run via `run_with_logging()`. Did the AI catch the unit issue?

### Key concepts

- Skills are NOT prompts. A prompt is what the user types now. A skill
  is what the AI knows about how to handle this *class* of task
  whenever it comes up.
- A good skill is the difference between *"the AI can handle this if I
  prompt carefully"* and *"any junior engineer can fire up Copilot and
  have the AI handle this correctly."*

### Proof of work [8 pts]

The text of your `unit_check_skill.md` printed via `Path.read_text()`,
a `run_with_logging()` transcript showing the AI being given the skill
as system prompt and catching the unit error, and a 2-sentence
observation contrasting "skill" vs. "prompt every time."

In [10]:
# Step 1 — Inspect the course skill at the repo root.
# Print the first ~40 lines and skim its structure.
from pathlib import Path
course_skill_path = Path("../../SKILL_me493b_course_structure.md")
if not course_skill_path.exists():
    course_skill_path = Path("SKILL_me493b_course_structure.md")
print("\n".join(course_skill_path.read_text(encoding="utf-8").splitlines()[:40]))


---
name: me493b-course-structure
description: >
  Reference for all structural, naming, and repository conventions for
  ME 493B "AI in Product Development" at UW Bothell (Spring 2026).
  Consult this file before creating any course file, notebook, folder,
  or spec document. Covers repo layout, file naming, notebook structure,
  spec document format, GitHub Classroom workflow, Canvas integration,
  and Copilot interaction patterns.
---

# ME 493B Course Structure Reference

## Course identity
- **Course:** ME 493B — AI in Product Development
- **Institution:** University of Washington Bothell
- **Term:** Spring 2026 (March 30 – June 5, 20 sessions, TTh 3:30–5:30 PM)
- **Instructor:** Scott Thielman, PhD (thielman@uw.edu)
- **Room:** UW1 041
- **Enrollment:** 24 students max
- **Credits:** 4

---

## Course architecture

### The Five Pillars (recurring vocabulary and rubric dimensions)

| Pillar | Short definition |
|--------|----------------|
| **Goal & Direction** | Specifying inten

In [11]:
# Step 2 — Read the specimen skill.
specimen_path = Path("skills/example_design_review.md")
if not specimen_path.exists():
    specimen_path = Path("MP3/Part A/skills/example_design_review.md")
print(specimen_path.read_text(encoding="utf-8"))

---
name: 3d-printed-part-design-review
description: >
  Review a 3D-printable mechanical part for FDM-printability, structural
  adequacy, and annotation hygiene before the part is sent to a print queue.
  Use when a user pastes a sketch, render, drawing, or description of a part
  that will be printed in PLA, PETG, or similar FDM material on a desktop
  machine (Prusa-class, Bambu-class, hobby Ender-class).
---

# Skill: 3D-Printed Part Design Review

## When to use this skill

Trigger this skill any time the user presents a part — image, screenshot,
verbal description, or CAD export — and asks for a review, critique, or
"what's wrong with this." Use it whenever the part will be printed via FDM.
Do NOT trigger for SLA, SLS, or machined parts; the constraints are
different.

## Steps

1. **Identify the part class.** Bracket, gear, housing, snap-fit, jig,
   hand tool, decorative? Different classes have different failure modes.
2. **Walk the print orientation.** What direction will the

### ✏️ YOUR TURN — Write `skills/unit_check_skill.md`

Create the file at `skills/unit_check_skill.md`. Include all five
required elements (front matter, when-to-use, steps, what-to-flag,
what-not-to-do). Aim for ~30–60 lines — sharp and specific beats long
and vague.

The next code cell prints your file back so the grader can see it. If
it errors with "FileNotFoundError," you haven't created the file yet.

In [12]:
# Step 3 — Print your skill back so it's part of the cell output.
skill_path = Path("skills/unit_check_skill.md")
if not skill_path.exists():
    skill_path = Path("MP3/Part A/skills/unit_check_skill.md")

print("=" * 62)
print(f"skills/unit_check_skill.md ({skill_path.resolve()})")
print("=" * 62)
if skill_path.exists():
    print(skill_path.read_text(encoding="utf-8"))
else:
    print("(file does not exist yet — create it, then re-run this cell)")

skills/unit_check_skill.md (/workspaces/ai-in-pd-spring2026-2/MP3/Part A/skills/unit_check_skill.md)
---
name: engineering-unit-conversion-review
description: >
  Scan engineering text (requirements, specs, reports, drawings notes)
  and flag likely unit-conversion errors, mixed-unit misuse, and
  numerically inconsistent values.
---

# Skill: Engineering Unit Conversion Review

## When to use this skill

Trigger this skill any time the user presents some text that contains different units and a conversion between them — and asks for a review, critique, or "what's wrong with this." 

## Steps

1. **Identify the Units Mentioned.** inch, psi, in-lb, lbf, °F? Different types of units have different uses and transformations

2. **Compute the Conversion between Units.** What should the units be when converted? What's the equivilent of the initial unit in another format?

3. **Confirm Document Conversion.** Are the conversions you made the same as the ones written on the document? Is there a

In [13]:
# Step 4 — Test the skill. Load it as a system prompt and send the test
# message. The AI should catch the unit error.
#
test_message = ("The shaft transmits 50 N·m of torque with a safety "
                 "factor of 2.5, requiring a diameter of 12 inches per "
                 "Lewis bending equation.")

result = run_with_logging(
     messages=[{"role": "user", "content": test_message}],
     system=skill_path.read_text(encoding="utf-8"),
     label="section2_skill_test",
 )


══════════════════════════════════════════════════════════════
[USER]
The shaft transmits 50 N·m of torque with a safety factor of 2.5, requiring a diameter of 12 inches per Lewis bending equation.
══════════════════════════════════════════════════════════════
[ITERATION 1]
[AI FINAL RESPONSE]
1. Mixed unit systems: The torque is specified in SI units (Newton-meter, N·m) while the required diameter is given in Imperial units (inches).  
   - Why it's a problem: For consistent engineering calculation, units should be uniform. Using N·m and inches together can lead to numerical errors unless correct conversion is performed.
   - Fix: Convert all units to either SI or Imperial before applying formulas (e.g., diameter in mm or torque in ft·lb).

2. Possible misuse of Lewis bending equation: The Lewis equation is primarily used for gear tooth strength, not for shaft diameter sizing under torque.  
   - Why it's a problem: Applying a formula intended for a different component can result in a

### ✏️ YOUR OBSERVATION

*In 2–3 sentences:* what's the difference between writing this as a
skill vs. just typing the instructions in your prompt every time?

> The main difference seems to be that skills like this can be reused for many scenarios, repeatedly giving the same output format. This repitition makes them far more useful and time consuming as you can find a skill detail that is easily understood and keep using that rather than simply typing into a prompt and getting slightly different results each time.

> **Forward connection.** In Section 5 you'll write a skill that
> orchestrates *multiple* tools. In Part B, your MiniClaw skills will
> be loaded by an AI host (Copilot agent mode, Claude Desktop, Cursor)
> with your MCP server connected — when a teammate asks a question
> about MiniClaw, your skill will guide the response.

---
## Section 3: Wrap Your RAG as a Tool [10 pts] — *centerpiece*

**Concept.** Your MP2 RAG is currently a Python function you call
directly. To make it AI-callable, you wrap it as a **tool** with a
JSON schema and connect it to the agent loop. The result: an AI that
can answer questions about Ridgeline's project history because it
knows to query your retrieval system *on demand*. This is the same
pattern that powers Copilot's codebase search, Claude's web search,
and every enterprise RAG product.

### What to build

1. The starter RAG provides `query_ridgeline_rag(question, n_results=3)`
   (already imported in the Setup section). If your MP2 RAG is fully
   working you can substitute it instead — but the starter is fine.
2. Wrap it as a tool with a JSON schema. **Critical:** the
   `description` field must be specific enough that the AI knows when
   to use it. Compare:
   - **Bad:** `"description": "Searches a database."`
   - **Good:** `"description": "Searches Ridgeline Engineering
     Partners' internal knowledge base — past project summaries,
     billing rates, technical standards, employee policies, FEA
     guidelines, and material selection reports. Use this when the
     user asks about Ridgeline's specific projects, internal
     practices, or company-specific information."`
3. Run three queries via `run_with_logging()`:
   - **Q1:** *"What was the rating factor on the Millbrook County bridge project?"*
   - **Q2:** *"What's the difference between cast iron and ductile iron?"*
     (general knowledge — should NOT call the tool)
   - **Q3:** *"What FEA mesh refinement standard does Ridgeline use,
     and is that consistent with industry best practice?"*
     (should call the tool for the first half, answer from training
     for the second half)
4. **Failure-mode exercise:** rewrite the tool description to be
   deliberately vague (e.g., `"Looks things up."`). Re-run Q1.
   Observe whether the AI still calls the tool. Restore the good
   description afterwards.
5. Build a summary table comparing `tool_called` vs. expected.

### Key concepts

- Tool description quality is the **dominant variable** in tool
  selection. Treat it like a Stack Overflow tag — specific enough that
  the right questions land here.
- The AI does NOT verify the tool's output. If your retrieval returns
  garbage, the AI will confidently use the garbage. Eval & Trust
  applies to tool results, not just final answers.
- Multi-turn behavior matters: the AI may call the tool, get a partial
  answer, and call it AGAIN with a refined query. That's the agent
  loop working.

### Proof of work [10 pts]

The full tool schema printed (good description), three labeled
transcripts for Q1/Q2/Q3, the failure-mode comparison transcript with
a vague description, a summary table, and a 3-sentence observation:
*"When did the AI choose to use your tool? When did it skip it? What
does this tell you about how to write tools for engineering work?"*

In [14]:
# Step 1 — Sanity-check that you can call the RAG directly.
# Print one chunk so you've seen the shape of the data.
sample = query_ridgeline_rag("standard billing rate for senior engineers", n_results=2)
print(sample["summary"])
for c in sample["chunks"]:
    print(f"  [{c['doc_id']}] {c['title']} (score {c['score']})")

2 chunks retrieved. Top match: DOC-014 — Standard Billing Rate Schedule — FY 2025 (score 0.6053).
  [DOC-014] Standard Billing Rate Schedule — FY 2025 (score 0.6053)
  [DOC-004] Employee Handbook Excerpts — Training and Development (score 0.4856)


In [15]:
# Step 2 — Write the tool schema for query_ridgeline_rag.
#
# Required parameter: question (string)
# Optional parameter: n_results (integer, default 3, min 1, max 10)
#
# Spend real effort on the description — make it sharp enough that the
# AI will pick this tool when it should and skip it when it shouldn't.
#
# rag_tool_good = { "type": "function", "function": { ... } }
# pretty_print_tool_schema(rag_tool_good)
rag_tool_good = {
    "type": "function",
    "function": {
        "name": "query_ridgeline_rag",
        "description": (
            "Answer questions about Ridgeline Engineering Partners by "
            "retrieving relevant information from the company's internal "
            "documents. The tool takes a natural language question and returns "
            "a concise summary along with the top relevant document chunks. "
            "Use this tool when the question is specifically about Ridgeline's "
            "practices, policies, or other internal information that an employee "
            "would know or could find out by looking at company documents."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "question": {
                    "type": "string",
                    "description": (
                        "The natural language question to ask about Ridgeline. Be as specific as possible."
                    )
                },
                "n_results": {
                    "type": "integer",
                    "description": (
                        "The number of relevant document chunks to return (1-10). Default is 3."
                    ),
                    "default": 3,
                    "minimum": 1,
                    "maximum": 10,
                },
            },
            "required": ["question"]
        }
    }
}
print("RAG tool schema defined:")
pretty_print_tool_schema(rag_tool_good)

RAG tool schema defined:
{
  "type": "function",
  "function": {
    "name": "query_ridgeline_rag",
    "description": "Answer questions about Ridgeline Engineering Partners by retrieving relevant information from the company's internal documents. The tool takes a natural language question and returns a concise summary along with the top relevant document chunks. Use this tool when the question is specifically about Ridgeline's practices, policies, or other internal information that an employee would know or could find out by looking at company documents.",
    "parameters": {
      "type": "object",
      "properties": {
        "question": {
          "type": "string",
          "description": "The natural language question to ask about Ridgeline. Be as specific as possible."
        },
        "n_results": {
          "type": "integer",
          "description": "The number of relevant document chunks to return (1-10). Default is 3.",
          "default": 3,
          "minimum": 1,
 

In [16]:
# Step 3 — Q1: Should call the tool
#   "What was the rating factor on the Millbrook County bridge project?"
#
# tool_handlers = {"query_ridgeline_rag": query_ridgeline_rag}
# result_q1 = run_with_logging(
#     messages=[{"role": "user", "content": "..."}],
#     tools=[rag_tool_good],
#     tool_handlers=tool_handlers,
#     label="section3_q1_millbrook",
# )
tool_handlers = {"query_ridgeline_rag": query_ridgeline_rag}
result_q1 = run_with_logging(
    messages=[{"role": "user", "content": "What was the rating factor on the Millbrook County bridge project?"}],
    tools=[rag_tool_good],
    tool_handlers=tool_handlers,
    label="section3_q1_millbrook",
)

══════════════════════════════════════════════════════════════
[USER]
What was the rating factor on the Millbrook County bridge project?
══════════════════════════════════════════════════════════════
[ITERATION 1]
[TOOL CALL] query_ridgeline_rag
  Args: {"question": "What was the rating factor on the Millbrook County bridge project?"}
[TOOL RESULT]
  {
    "chunks": [
      {
        "text": "PROJECT SUMMARY REPORT\nProject: Millbrook County DOT Bridge Load Rating Assessment\nProject Number: REP-2023-047\nClient: Millbrook County Department of Transportation\nProject Manager: Dr. Sarah Hendricks, PE, PhD\nDate Completed: August 2024\nBudget: $285,000 | Actual: $271,400\n\nSCOPE\nRidgeline was engaged by the Millbrook County DOT to perform a comprehensive load rating assessment of the Cedar Creek Bridge (Bridge No. MC-4412), a three-span continuous steel plate girder bridge constructed in 1987",
        "doc_id": "DOC-006",
        "title": "Project  ...
════════════════════════════════

In [19]:
# Step 3 — Q2: Should NOT call the tool (general knowledge)
#   "What's the difference between cast iron and ductile iron?"
# label = "section3_q2_cast_iron"
result_q2 = run_with_logging(
    messages=[{"role": "user", "content": "What's the difference between cast iron and ductile iron?"}],
    tools=[rag_tool_good],
    tool_handlers=tool_handlers,
    label="section3_q2_cast_iron",
)

══════════════════════════════════════════════════════════════
[USER]
What's the difference between cast iron and ductile iron?
══════════════════════════════════════════════════════════════
[ITERATION 1]
[AI FINAL RESPONSE]
**Cast iron** and **ductile iron** are two types of iron materials used in engineering and manufacturing, but they differ in their composition, properties, and applications.

### Cast Iron
- **Composition:** Cast iron is an alloy of iron, carbon (2–4%), and silicon.
- **Microstructure:** It contains flakes of graphite, which make it brittle.
- **Properties:**
  - Hard and brittle (breaks easily under tension)
  - Good compressive strength but poor tensile strength
  - Not easily bendable or malleable
  - Excellent machinability and resistance to wear
- **Uses:** Engine blocks, pipes, cookware, machine bases, frames.

### Ductile Iron (also known as Nodular Iron or Spheroidal Graphite Iron)
- **Composition:** Similar to cast iron but with some magnesium added.
- **M

In [20]:
# Step 3 — Q3: Multi-turn judgment call
#   "What FEA mesh refinement standard does Ridgeline use, and is that
#    consistent with industry best practice?"
# label = "section3_q3_fea_mesh"
result_q3 = run_with_logging(
    messages=[{"role": "user", "content": (
        "What FEA mesh refinement standard does Ridgeline use, and is that "
        "consistent with industry best practice?"
    )}],
    tools=[rag_tool_good],
    tool_handlers=tool_handlers,
    label="section3_q3_fea_mesh",
)

══════════════════════════════════════════════════════════════
[USER]
What FEA mesh refinement standard does Ridgeline use, and is that consistent with industry best practice?
══════════════════════════════════════════════════════════════
[ITERATION 1]
[TOOL CALL] query_ridgeline_rag
  Args: {"n_results": 3, "question": "What FEA mesh refinement standard does Ridgeline use?"}
[TOOL RESULT]
  {
    "chunks": [
      {
        "text": "RIDGELINE ENGINEERING PARTNERS — FEA BEST PRACTICES GUIDE\nDocument Number: ENG-FEA-BP-01, Revision 4\nEffective Date: April 2024\nAuthor: Dr. Sarah Hendricks, PE, PhD\nApproved by: Brian Tanaka, PE, Director of Engineering\n\nThis guide establishes minimum requirements for finite element analysis performed at Ridgeline. All FEA work must comply with these practices regardless of software platform (ANSYS, Abaqus, NASTRAN, CSiBridge, COMSOL, etc.).\n\n1. MODEL PLANNING\nBefore constructing any FEA model, ",
        "doc_id": "DOC-017",
        "title": "FEA

In [21]:
# Step 4 — Failure-mode exercise.
#
# Make a copy of your tool schema with a deliberately vague description
# (something like "Looks things up.") and re-run Q1. Compare to the
# good-description run.
#
# rag_tool_vague = { ... }   # same name, vague description
# result_q1_vague = run_with_logging(
#     messages=[{"role": "user", "content": "What was the rating factor on the Millbrook County bridge project?"}],
#     tools=[rag_tool_vague],
#     tool_handlers=tool_handlers,
#     label="section3_q1_vague_description",
# )
rag_tool_vague = {
    "type": "function",
    "function": {
        "name": "query_ridgeline_rag",
        "description": "Looks things up.",
        "parameters": {
            "type": "object",
            "properties": {
                "question": {"type": "string"},
                "n_results": {"type": "integer", "default": 3, "minimum": 1, "maximum": 10},
            },
            "required": ["question"]
        }
    }
}
result_q1_vague = run_with_logging(
    messages=[{"role": "user", "content": "What was the rating factor on the Millbrook County bridge project?"}],
    tools=[rag_tool_vague],
    tool_handlers=tool_handlers,
    label="section3_q1_vague_description",
)


══════════════════════════════════════════════════════════════
[USER]
What was the rating factor on the Millbrook County bridge project?
══════════════════════════════════════════════════════════════
[ITERATION 1]
[TOOL CALL] query_ridgeline_rag
  Args: {"question": "What was the rating factor on the Millbrook County bridge project?"}
[TOOL RESULT]
  {
    "chunks": [
      {
        "text": "PROJECT SUMMARY REPORT\nProject: Millbrook County DOT Bridge Load Rating Assessment\nProject Number: REP-2023-047\nClient: Millbrook County Department of Transportation\nProject Manager: Dr. Sarah Hendricks, PE, PhD\nDate Completed: August 2024\nBudget: $285,000 | Actual: $271,400\n\nSCOPE\nRidgeline was engaged by the Millbrook County DOT to perform a comprehensive load rating assessment of the Cedar Creek Bridge (Bridge No. MC-4412), a three-span continuous steel plate girder bridge constructed in 1987",
        "doc_id": "DOC-006",
        "title": "Project  ...
════════════════════════════════

In [22]:
# Step 5 — Summary table.
#
# Print:
#   Q1 (Millbrook bridge):       tool_called = ?   expected = yes
#   Q2 (cast vs ductile iron):   tool_called = ?   expected = no
#   Q3 (FEA mesh + industry):    tool_called = ?   expected = yes
#   Q1 with vague description:   tool_called = ?   (your prediction)
print("Summary of RAG tool calls:")
print(f"  Q1 (Millbrook bridge):       tool_called = {'yes' if result_q1['tool_calls_made'] else 'no'}   expected = yes")
print(f"  Q2 (cast vs ductile iron):   tool_called = {'yes' if result_q2['tool_calls_made'] else 'no'}   expected = no")
print(f"  Q3 (FEA mesh + industry):    tool_called = {'yes' if result_q3['tool_calls_made'] else 'no'}   expected = yes")
print(f"  Q1 with vague description:   tool_called = {'yes' if result_q1_vague['tool_calls_made'] else 'no'}   expected = no")

Summary of RAG tool calls:
  Q1 (Millbrook bridge):       tool_called = yes   expected = yes
  Q2 (cast vs ductile iron):   tool_called = no   expected = no
  Q3 (FEA mesh + industry):    tool_called = yes   expected = yes
  Q1 with vague description:   tool_called = yes   expected = no


### ✏️ YOUR OBSERVATION

*In 3 sentences:* When did the AI choose to use your tool? When did it
skip it? What does this tell you about how to write tools for
engineering work?

> It chose to use the tool anytime numbers or specific measurements came up, ignoring the tool only when general knowledge was required instead of looking at a specific situation. This tells me that many tools are very specific in their usage, applying to anything even possibly related to their topic, and if you want a program to use them you must phrase that initial when to apply list carefully. 

> **Forward connection.** In Part B you'll wrap your *MiniClaw* RAG as
> an MCP server tool that Copilot or Claude Desktop can call from your
> IDE. The principle is identical to what you just did — only the
> wire protocol changes.

---
## Section 4: Vision in the Loop — Host-Based Design Review [8 pts]

**Concept.** Engineering work is visual. CAD models, sketches, hand
drawings, prototype photos, render captures — most engineering
judgment happens *by looking at things*. Modern AI hosts (Copilot Chat,
Claude.ai, Claude Desktop, Cursor, ChatGPT) all accept image input
directly in the chat interface. This is the workflow you'll actually
use as a working engineer: paste a screenshot, ask for review, decide
what to act on. **No API plumbing required.**

The proof-of-work for this section is a *documented exchange* —
markdown + screenshot + response text. Same evidence pattern Part B
uses for the CAD interaction. Section 4 is your warm-up.

### What to do

1. Open the AI host of your choice. **Copilot Chat in VS Code is the
   default** — it's already installed and you've been using it. Other
   acceptable hosts: Claude.ai (web), Claude Desktop, Cursor,
   ChatGPT. Pick one and note it.
2. Load `images/bracket_sketch.png` (a hand-sketched L-bracket with
   deliberate errors). Paste it into the host's chat.
3. Send the prompt:

   > *"You are reviewing a hand sketch for an L-bracket that will be
   > 3D printed in PLA. Identify any issues you see — dimensional,
   > manufacturing, structural, or annotation. Be specific."*
4. Capture three things:
   - A markdown cell with the prompt you sent
   - A screenshot of the host UI showing the image and the response,
     saved to `section4_screenshots/sketch_review.png` (or similar)
   - A markdown cell with the response text copied in
5. Repeat with `images/openscad_render.png` (a render with a visible
   undercut that won't print well). Same prompt, same evidence
   capture.
6. **Comparative exercise.** Open a fresh chat and ask the same review
   question with NO image — just a text description: *"I have a
   hand-sketched L-bracket for 3D printing in PLA. Annotated
   dimensions, two M4 holes, gusset on the back. What issues should I
   check for?"* Capture the response. Compare to the image-based
   reviews.

### Key concepts

- This is the workflow you'll actually use day-to-day, and it's
  exactly what Part B's CAD interaction deliverable asks for.
- Hosts handle image encoding, model selection, and API plumbing for
  you. Your job is the engineering question and the engineering
  judgment.
- The AI's vision is good but not perfect. It will sometimes
  hallucinate features, miss subtle issues, or misread numbers. This
  is exactly where Eval & Trust applies — *the engineer judges what
  to act on.*
- Different hosts use different vision-capable models with different
  strengths. If a review feels thin, try a second host.

### Proof of work [8 pts]

For each of the **three exchanges** (sketch image, CAD render image,
text-only baseline), commit:
- The prompt as markdown
- A screenshot in `section4_screenshots/`
- The response text as markdown

Plus the comparison table below — fill in the issue catalog and the
counts.

### Exchange 1 — Hand sketch (bracket_sketch.png)

**Host used:** Copilot

**Prompt I sent:**

> You are reviewing a hand sketch for an L-bracket that will be
   > 3D printed in PLA. Identify any issues you see — dimensional,
   > manufacturing, structural, or annotation. Be specific.
   > Condense your feedback to fit in an image


**Screenshot:** `section4_screenshots/sketch_review.png`

![Sketch review](section4_screenshots/sketch_review.png)



**Response (copied from host):**

> ⚙️ Dimensional and Annotation Issues
> Mixed units: The drawing uses both inches (2.36") and millimeters (80 mm, 32 mm, 4 mm). Mixing units can cause confusion or scaling errors during slicing or CAD modeling. Choose one consistent unit system.

> Tolerance mismatch: The note “ALL DIMS ±0.001″” is unrealistic for FDM printing. Typical PLA tolerances are around ±0.2 mm (≈ ±0.008″). The stated precision is closer to CNC machining, not additive manufacturing.

> Wall thickness: A 0.3 mm wall is far too thin for PLA FDM printing. Most printers require at least 0.8–1.2 mm (2–3 perimeters) for structural integrity.

> Hole dimension clarity: The 4 mm holes are not dimensioned relative to edges or the bracket’s height. Without full positional dimensions, alignment and fit cannot be verified.

> Annotation style: “Deliberately rough” and “hand sketch” are fine for concept work, but production drawings should include clear line types, dimension arrows, and consistent font size.

> 🧱 Structural and Manufacturing Concerns
> Thin vertical wall: At 0.3 mm, the upright section will likely warp or snap under minimal load. Increase thickness or add fillets/ribs for stiffness.

> Unsupported overhang: The inside corner may require support material during printing. Consider chamfering or adding a gusset to reduce stress and improve printability.

> Hole bridging: The holes near the base could cause bridging issues if printed horizontally. Orient the part or add chamfers to improve surface finish.

> Material choice: PLA is brittle; for brackets under load, PETG or ABS may perform better.

> 🧩 Recommendations
> Use consistent metric units throughout.

> Revise tolerances to ±0.2 mm.

> Increase wall thickness to ≥ 1 mm.

> Add fillets or gussets for strength.

> Include complete hole positioning dimensions.

> Consider alternative materials for durability.

### Exchange 2 — OpenSCAD render (openscad_render.png)

**Host used:** Copilot

**Prompt I sent:**

> You are reviewing a hand sketch for an L-bracket that will be
   > 3D printed in PLA. Identify any issues you see — dimensional,
   > manufacturing, structural, or annotation. Be specific.
   > Condense your feedback to fit in an image

**Screenshot:** `section4_screenshots/render_review_2.png`

![Sketch review](section4_screenshots/sketch_review_2.png)

**Response (copied from host):**

> Here’s a condensed visual summary of the OpenSCAD L‑bracket v2 review — an annotated image highlighting the key improvements and remaining issues:

> 🟢 Triangular gusset: Reinforces the joint, reducing flex under load.

> ⚙️ Consistent metric units: Eliminates scaling errors from mixed dimensions.

> 🧱 Optimized for FDM printing: Works well with 0.4 mm nozzle, 0.2 mm layers; orient gusset upward.

> 🔴 Sharp inner corner: Still a stress concentration; add ≈ 2 mm fillet.

> 🔵 Hole sizing: Verify clearance; oversize by +0.2 mm for PLA shrinkage.

### Exchange 3 — Text-only baseline (no image)

**Host used:** Copilot

**Prompt I sent:**

> *"I have a hand-sketched L-bracket for 3D printing in PLA. Annotated
> dimensions, two M4 holes, gusset on the back. What issues should I
> check for?"*

**Screenshot:** `section4_screenshots/textonly_review_3.png`

![Sketch review](section4_screenshots/sketch_review_3.png)

**Response (copied from host):**

> 🧱 Structural checks
Wall thickness — Ensure all load‑bearing walls are ≥ 3 mm for PLA. Thin walls near the M4 holes can crack under torque.

Gusset geometry — A gusset helps, but check that it actually intersects both arms with enough surface area. Shallow or narrow gussets add little stiffness.

Stress concentration — Sharp inner corners are classic failure points. Add a 2–3 mm fillet where the arms meet.

Load direction — Confirm the bracket’s strongest axis aligns with the expected load. PLA is brittle when stressed across layers.

🔩 Fastener‑related issues
Hole clearance — M4 clearance should be 4.2–4.4 mm in PLA. Undersized holes cause cracking when inserting screws.

Countersinks or counterbores — If you need flush screws, plan these early; retrofitting them after slicing often weakens the part.

Edge distance — Keep screw holes at least 1.5× diameter from any edge to avoid splitting.

🖨️ Printability concerns
Print orientation — The bracket should be oriented so the layer lines run along the arms, not across the bend. Otherwise it snaps at the corner.

Support requirements — Gussets often create overhangs. Check whether your design forces heavy supports that scar the surface.

Warping and shrinkage — PLA shrinks slightly; long arms may curl upward. Add chamfers or brims if needed.

📏 Dimensional clarity
Consistent units — If you move this into CAD or OpenSCAD, ensure everything is in millimeters. Mixed units cause scaling disasters.

Tolerance planning — Any mating surfaces or sliding fits need ~0.2–0.3 mm clearance.

### Comparison table

Fill in the issue catalog (your own ground-truth list of what's wrong
with the bracket — you should be able to identify 3–5 distinct issues
just from the sketch and render). Then count what each AI exchange
caught.

**My ground-truth issue list:**

1. Different measurement units, Inches and mm should not be used on a single sketch
2. Sharp Angle with thin walls, would be very easy to break off under pressure, smooth fillet or added support at corner
3. +/- 0.001 is way to tight a tolerance to be default, that is only used for very specific and highly sensitive areas on small devices, not a basic bracket
4. No initial distance from the leftmost hole to the wall
5. No size given for the size of the holes.

| Exchange                | Host | Issues caught | Out of |
|-------------------------|------|---------------|--------|
| Sketch image            |      |               |        |
| OpenSCAD render image   |      |               |        |
| Text-only baseline      |      |               |        |

### ✏️ YOUR OBSERVATION

*In 3 sentences:* Where does vision help? Where does it fall short?
What would you NOT trust to a vision review?

> _Replace this line with your answer._

> **Forward connection.** In Part B you'll do this exact workflow on
> your own MiniClaw CAD work — three documented checkpoints, a
> screenshot to your AI host, a documented review. Section 4 was the
> warm-up; Part B is the real thing.

---
## Section 5: Composition Capstone — A Skill That Uses Multiple Tools [8 pts]

**Concept.** A real engineering AI workflow combines multiple
capabilities. The capstone exercise: write a skill that orchestrates
the AI to consult relevant project history (your **RAG tool from
Section 3**) *and* handle unit conversions (your **converter from
Section 1**) in the same response, then produce a synthesized
engineering answer drawing on both. The AI decides which tool to call
when, guided by the skill.

### What to build

1. Write `skills/engineering_question_with_context.md`. The skill
   should instruct the AI to:
   - First query the RAG for relevant project history when the
     question touches Ridgeline projects
   - Use the unit converter whenever a quantity in one unit system
     needs to be expressed in another
   - **Synthesize:** tie retrieved context to the specific question,
     don't dump retrieved chunks
2. Load the skill as system prompt. Make BOTH the RAG tool and the
   unit converter available.
3. Run a composite query through `run_with_logging()`:

   > *"I'm reviewing a bracket design for a project similar to your
   > CardioSense work. The bracket needs to handle a 500 lbf load.
   > What did Ridgeline use for the CardioSense enclosure, and what's
   > that load in newtons?"*
4. **Without the skill:** rerun the same query with only the tools
   (no system prompt). Compare. What did the skill change?

### Key concepts

- This is the agent loop with multiple tools — the architectural
  pattern that powers Claude Code, Copilot agent mode, Cursor agent,
  and every "AI coworker" product on the market.
- The skill is what turns *"AI with tools"* into *"AI with judgment
  about which tools to use when."* Without the skill, the AI may
  answer from training without consulting the RAG, may convert units
  mentally, or may not synthesize across both.
- Composition introduces the failure mode of *under-consultation* —
  the AI deciding it doesn't need a tool when it does. The skill
  counters this by making consultation mandatory in the workflow.

### Proof of work [8 pts]

The text of `engineering_question_with_context.md`, two transcripts
(with skill / without skill), a summary table, and a 3-sentence
observation: *"What does a skill add to a tool-equipped AI? What's
the limit of what skills can do?"*

### ✏️ YOUR TURN — Write `skills/engineering_question_with_context.md`

Same format as your Section 2 skill (front matter, when-to-use, steps,
what-to-flag, what-not-to-do). Include explicit instructions about
*when* to call each tool.

In [ ]:
# Step 1 — Print your composition skill so it's part of the cell output.
composite_skill_path = Path("skills/engineering_question_with_context.md")
if not composite_skill_path.exists():
    composite_skill_path = Path("MP3/Part A/skills/engineering_question_with_context.md")

print("=" * 62)
print(f"{composite_skill_path.name}")
print("=" * 62)
if composite_skill_path.exists():
    print(composite_skill_path.read_text(encoding="utf-8"))
else:
    print("(file does not exist yet — create it, then re-run this cell)")

In [ ]:
# Step 2 — WITH skill: load skill as system prompt, expose both tools.
#
# composite_query = ("I'm reviewing a bracket design for a project similar "
#                    "to your CardioSense work. The bracket needs to handle "
#                    "a 500 lbf load. What did Ridgeline use for the "
#                    "CardioSense enclosure, and what's that load in newtons?")
#
# tools = [rag_tool_good, convert_units_tool]
# tool_handlers = {
#     "query_ridgeline_rag": query_ridgeline_rag,
#     "convert_units":       convert_units,
# }
#
# result_with_skill = run_with_logging(
#     messages=[{"role": "user", "content": composite_query}],
#     tools=tools,
#     tool_handlers=tool_handlers,
#     system=composite_skill_path.read_text(encoding="utf-8"),
#     label="section5_with_skill",
# )

In [ ]:
# Step 3 — WITHOUT skill: same query, same tools, no system prompt.
# result_without_skill = run_with_logging(
#     messages=[{"role": "user", "content": composite_query}],
#     tools=tools,
#     tool_handlers=tool_handlers,
#     label="section5_without_skill",
# )

In [ ]:
# Step 4 — Summary table.
#
# Print something like:
#   With skill:    rag_called = ?   converter_called = ?   synthesis = present/absent
#   Without skill: rag_called = ?   converter_called = ?   synthesis = present/absent
#
# Use result_with_skill["tool_calls_made"] and result_without_skill["tool_calls_made"].

### ✏️ YOUR OBSERVATION

*In 3 sentences:* What does a skill add to a tool-equipped AI? What's
the limit of what skills can do?

> _Replace this line with your answer._

> **Forward connection.** Part B asks you to do this on your own
> engineering work — write skills, expose MiniClaw RAG as an MCP
> tool, and run a documented design review. The composition you just
> built is the seed of that workflow at MiniClaw scale.

---
## Section 6: Reflections [8 pts]

Three short reflections. 3–4 sentences each, with at least one
concrete engineering example where it makes sense. The grader is
looking for thought, not length.

### Reflection 1 — Where do tools end and skills begin?

In your own words, when would you reach for a new *tool* vs. a new
*skill*? Give a small engineering example for each.

> _Your answer here._

### Reflection 2 — What's the durable lesson?

MCP, function calling, GitHub Models, host-based vision review —
these are 2026 specifics. What's the underlying idea you'd carry
forward if all of these were replaced next year?

> _Your answer here._

### Reflection 3 — What worries you?

When AI orchestrates tools and consults skills, the engineer steps
back from doing the work directly. What's the failure mode you'd
watch for in a real engineering team?

> _Your answer here._

---

## Submission checklist

Before pushing your final commit, verify:

- [ ] All cells in Sections 1, 2, 3, 5 run end-to-end without errors
- [ ] At least one `[TOOL CALL]` line is visible in each of Sections 1, 3, 5
- [ ] `skills/unit_check_skill.md` and
      `skills/engineering_question_with_context.md` both exist and were
      printed as part of the cell output
- [ ] Section 4 has three documented exchanges with screenshots
      committed in `section4_screenshots/`
- [ ] `transcripts/` contains the JSON files written by
      `run_with_logging()` (these get committed automatically when you
      push)
- [ ] Section 6 reflections are filled in

> **Codespaces save warning.** Make sure the notebook is saved
> (`Ctrl+S`) and committed via Source Control before you stop your
> Codespace. Unstaged changes are lost when the container is deleted.